In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import pandas as pd

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Load Dataset (MNIST)

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)


100%|██████████| 9.91M/9.91M [00:00<00:00, 22.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 623kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 5.69MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.23MB/s]


Define Simple CNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, 1)
        self.conv2 = nn.Conv2d(16, 32, 3, 1)
        self.fc1 = nn.Linear(32*5*5, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)


Function to Train & Evaluate

In [ ]:
def train_eval(model, optimizer, epochs=3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    train_acc_list = []

    for epoch in range(epochs):
        model.train()
        correct, total = 0, 0
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc_list.append(100 * correct / total)

    # Test Accuracy
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_acc = 100 * correct / total
    return train_acc_list[-1], test_acc


In [ ]:
optimizers = {
    "SGD": optim.SGD(model.parameters(), lr=0.01),
    "SGD+Momentum": optim.SGD(model.parameters(), lr=0.01, momentum=0.9),
    "NAG": optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True),
    "Adagrad": optim.Adagrad(model.parameters(), lr=0.01),
    "RMSprop": optim.RMSprop(model.parameters(), lr=0.001),
    "Adam": optim.Adam(model.parameters(), lr=0.001)
}

results = []

for name, opt in optimizers.items():
    print(f"Training with {name} ...")
    model = SimpleCNN().to(device)  # Reset model for fair comparison
    train_acc, test_acc = train_eval(model, opt, epochs=3)
    results.append([name, train_acc, test_acc])

# Show results as a table
df = pd.DataFrame(results, columns=["Optimizer", "Train Accuracy (%)", "Test Accuracy (%)"])
print(df)


Training with SGD ...
Training with SGD+Momentum ...
Training with NAG ...
Training with Adagrad ...
Training with RMSprop ...
Training with Adam ...
      Optimizer  Train Accuracy (%)  Test Accuracy (%)
0           SGD           10.481667              11.16
1  SGD+Momentum            7.511667               6.76
2           NAG           10.301667              10.56
3       Adagrad            8.740000               8.36
4       RMSprop           11.198333              11.37
5          Adam           10.780000              10.99
